# Overfitting & Underfitting with PyTorch

This notebook demonstrates overfitting and underfitting on MNIST and shows how to fix them.

**Topics covered:**
- What is Underfitting? (model too simple)
- What is Overfitting? (model memorizes training data)
- Regularization techniques: **Dropout**, **L2 Weight Decay**, **Early Stopping**


## Setup


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np

NUM_EPOCHS    = 5
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {DEVICE}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  BATCH_SIZE, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Testing samples  : {len(test_dataset):,}")


## Helper Functions


In [ ]:
loss_function = nn.CrossEntropyLoss()

def train_model(model, device, loader, optimizer, epoch):
    model.train()
    total_loss, correct = 0, 0
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = loss_function(outputs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == target).sum().item()
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} | Step {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")
    return total_loss / len(loader), 100.0 * correct / len(loader.dataset)


def evaluate_model(model, device, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += loss_function(output, target).item()
            correct    += (output.argmax(dim=1) == target).sum().item()
    return total_loss / len(loader), 100.0 * correct / len(loader.dataset)


def plot_training_results(train_losses, test_losses, train_accs, test_accs, title_prefix=""):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="Train Loss",  color="blue")
    plt.plot(test_losses,  label="Test Loss",   color="red")
    plt.title(f"{title_prefix} — Loss Curves")
    plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label="Train Acc", color="blue")
    plt.plot(test_accs,  label="Test Acc",  color="red")
    plt.title(f"{title_prefix} — Accuracy Curves")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy %"); plt.legend()

    plt.tight_layout(); plt.show()


## Part 1 — Underfitting

**Underfitting** happens when the model is too simple to capture the patterns in the data.  
Signs: both training and test loss are high; they stay close together.

We force underfitting by using a tiny hidden layer (only 5 neurons for a 784-dimensional input).


In [ ]:
class UnderfitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 5),   # ← bottleneck! 784 → 5
            nn.ReLU(),
            nn.Linear(5, 10)
        )
    def forward(self, x):
        return self.net(self.flatten(x))


underfit_model     = UnderfitModel().to(DEVICE)
underfit_optimizer = optim.Adam(underfit_model.parameters(), LEARNING_RATE)

u_train_losses, u_test_losses = [], []
u_train_accs,   u_test_accs   = [], []

print("Training underfitting model...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_model(underfit_model, DEVICE, train_loader, underfit_optimizer, epoch)
    te_loss, te_acc = evaluate_model(underfit_model, DEVICE, test_loader)
    u_train_losses.append(tr_loss); u_test_losses.append(te_loss)
    u_train_accs.append(tr_acc);    u_test_accs.append(te_acc)
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_training_results(u_train_losses, u_test_losses, u_train_accs, u_test_accs, "Underfitting")


**Analysis:**  
Notice how both train and test accuracy are stuck at a low ceiling (~90%).  
The 5-neuron bottleneck prevents the model from learning complex digit features — that's underfitting.


## Part 2 — Overfitting

**Overfitting** happens when the model memorizes the training data instead of generalizing.  
Signs: training accuracy keeps rising while test accuracy plateaus or drops.

We force overfitting by: (1) using a large model with many parameters, and (2) training on only 10% of the data.


In [ ]:
# Train on a small subset to make overfitting easier to observe
overfit_subset = Subset(train_dataset, range(6000))
overfit_loader = DataLoader(overfit_subset, batch_size=32, shuffle=True)

class OverfitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 1024), nn.ReLU(),
            nn.Linear(1024, 512),   nn.ReLU(),
            nn.Linear(512, 512),    nn.ReLU(),
            nn.Linear(512, 10)
            # No Dropout → will overfit
        )
    def forward(self, x):
        return self.net(self.flatten(x))


overfit_model     = OverfitModel().to(DEVICE)
overfit_optimizer = optim.Adam(overfit_model.parameters(), lr=1e-4)

of_train_losses, of_test_losses = [], []
of_train_accs,   of_test_accs   = [], []

print("Training overfitting model (20 epochs on 10% data)...")
for epoch in range(1, 21):
    tr_loss, tr_acc = train_model(overfit_model, DEVICE, overfit_loader, overfit_optimizer, epoch)
    te_loss, te_acc = evaluate_model(overfit_model, DEVICE, test_loader)
    of_train_losses.append(tr_loss); of_test_losses.append(te_loss)
    of_train_accs.append(tr_acc);    of_test_accs.append(te_acc)
    if epoch % 5 == 0:
        print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_training_results(of_train_losses, of_test_losses, of_train_accs, of_test_accs, "Overfitting")


## Part 3 — Fix: Dropout

**Dropout** randomly zeroes a fraction of neurons during training.  
This prevents neurons from co-adapting and forces the network to learn more robust features.


In [ ]:
class RegularizedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 1024), nn.ReLU(),
            nn.Linear(1024, 512),   nn.ReLU(),
            nn.Linear(512, 512),    nn.ReLU(),
            nn.Dropout(p=0.5),      # ← randomly drops 50% of neurons
            nn.Linear(512, 10)
        )
    def forward(self, x):
        return self.net(self.flatten(x))


model_dropout     = RegularizedModel().to(DEVICE)
dropout_optimizer = optim.Adam(model_dropout.parameters(), lr=1e-4)

dr_train_losses, dr_test_losses = [], []
dr_train_accs,   dr_test_accs   = [], []

print("Training with Dropout...")
for epoch in range(1, 21):
    tr_loss, tr_acc = train_model(model_dropout, DEVICE, overfit_loader, dropout_optimizer, epoch)
    te_loss, te_acc = evaluate_model(model_dropout, DEVICE, test_loader)
    dr_train_losses.append(tr_loss); dr_test_losses.append(te_loss)
    dr_train_accs.append(tr_acc);    dr_test_accs.append(te_acc)
    if epoch % 5 == 0:
        print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_training_results(dr_train_losses, dr_test_losses, dr_train_accs, dr_test_accs, "With Dropout")


## Part 4 — Fix: L2 Regularization (Weight Decay)

**L2 Regularization** adds a penalty proportional to the square of the weights to the loss.  
This discourages large weights and keeps the model simpler.  
In PyTorch, it's added via the `weight_decay` parameter in the optimizer.


In [ ]:
model_l2    = RegularizedModel().to(DEVICE)
l2_optimizer = optim.Adam(model_l2.parameters(), lr=1e-3, weight_decay=1e-4)  # ← L2 here

l2_train_losses, l2_test_losses = [], []
l2_train_accs,   l2_test_accs   = [], []

print("Training with L2 + Dropout...")
for epoch in range(1, 21):
    tr_loss, tr_acc = train_model(model_l2, DEVICE, overfit_loader, l2_optimizer, epoch)
    te_loss, te_acc = evaluate_model(model_l2, DEVICE, test_loader)
    l2_train_losses.append(tr_loss); l2_test_losses.append(te_loss)
    l2_train_accs.append(tr_acc);    l2_test_accs.append(te_acc)
    if epoch % 5 == 0:
        print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_training_results(l2_train_losses, l2_test_losses, l2_train_accs, l2_test_accs, "L2 + Dropout")


## Part 5 — Comparison


In [ ]:
plt.figure(figsize=(14, 6))

# Test accuracy comparison
plt.subplot(1, 2, 1)
plt.plot(of_test_accs, label="No Regularization", color="red",   linestyle="--")
plt.plot(dr_test_accs, label="Dropout",            color="blue")
plt.plot(l2_test_accs, label="L2 + Dropout",       color="green")
plt.title("Test Accuracy Comparison")
plt.xlabel("Epochs"); plt.ylabel("Accuracy %"); plt.legend()

# Overfitting gap (train - test)
plt.subplot(1, 2, 2)
plt.plot(np.array(of_train_accs) - np.array(of_test_accs), label="No Reg",      color="red",   linestyle="--")
plt.plot(np.array(dr_train_accs) - np.array(dr_test_accs), label="Dropout",     color="blue")
plt.plot(np.array(l2_train_accs) - np.array(l2_test_accs), label="L2+Dropout",  color="green")
plt.title("Overfitting Gap (Train Acc − Test Acc)")
plt.xlabel("Epochs"); plt.ylabel("% gap"); plt.legend()

plt.tight_layout(); plt.show()

print("\nKey takeaway:")
print("  • No regularization → large gap between train and test (memorization)")
print("  • Dropout / L2       → smaller gap, better generalization")
